# Track-A v1.2 — MASTER K3 (v5)

Two-GPU worker. Acquires canonical K1 G1A, runs CAL-R13 on GPU0 during G2A, waits for the sealed control plane, then uses both GPUs during K3 science.

Use **Kaggle T4 x2**, **Internet ON**, and **Save Version -> Save & Run All / Batch**.

**Frozen science SHA:** `9a72e9466a9a3e7429e0e36a028edac662f83146`  
**Pinned operator runtime:** `902e7774dda32106a03bfb3f5917946c11ff8e2c`  
**Runtime branch:** `ops-tracka-kaggle-master-runtime-v5-902e777`

Attach the frozen CropCop V1 dataset. The K1 private G1A dataset must be shared with this Kaggle account before the G1A handoff becomes usable.

Required Kaggle secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `CROPCOP_GITHUB_TOKEN`.

This v5 notebook prefers the exact V1 mount layout already proven in the real Kaggle run, but every preferred path is re-verified by the runtime against frozen hashes and TRAIN/VAL structure. If the mount prefix changes, the bounded resolver falls back automatically.

Do not edit scientific settings in this notebook. All experiment identity, calibration mapping, scheduler, durability, GO, and training semantics remain in the frozen scientific source/runtime.


In [ ]:
from pathlib import Path
import os

# Preferred paths proven by the real K1 Kaggle mount. The runtime still hash/structure-verifies them.
V1 = Path('/kaggle/input/datasets/ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1/CropCop_Final_v1')
if V1.is_dir():
    preferred = {
        'CROPCOP_MANIFEST': V1 / 'audit' / 'final_manifest.csv',
        'CROPCOP_CLASS_MAP': V1 / 'audit' / 'class_to_idx.json',
        'CROPCOP_IMAGE_ROOT': V1 / 'dataset',
    }
    for name, path in preferred.items():
        if path.exists():
            os.environ.setdefault(name, str(path))
    print('Preferred frozen V1 mount detected:', V1)
else:
    print('Preferred V1 mount prefix not present; bounded hash/structure resolver will be used.')

# Optional: bind this logical lane to one expected Kaggle username.
# os.environ['CROPCOP_EXPECTED_KAGGLE_USERNAME'] = 'your-kaggle-username'

# Optional fail-closed overrides only if the runtime reports a genuine ambiguity.
# os.environ['CROPCOP_MANIFEST'] = '/kaggle/input/.../final_manifest.csv'
# os.environ['CROPCOP_CLASS_MAP'] = '/kaggle/input/.../class_to_idx.json'
# os.environ['CROPCOP_IMAGE_ROOT'] = '/kaggle/input/.../dataset'


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

OPS_RUNTIME_SHA = '902e7774dda32106a03bfb3f5917946c11ff8e2c'
OPS_RUNTIME_BRANCH = 'ops-tracka-kaggle-master-runtime-v5-902e777'
OPS_ROOT = Path('/kaggle/working/cropcop-tracka-master-runtime')

if OPS_ROOT.exists():
    shutil.rmtree(OPS_ROOT)

subprocess.run([
    'git', 'clone', '--quiet', '--depth', '1', '--branch', OPS_RUNTIME_BRANCH,
    'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git', str(OPS_ROOT)
], check=True)

head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=OPS_ROOT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=OPS_ROOT, text=True).strip()

if head != OPS_RUNTIME_SHA:
    raise RuntimeError(f'operator runtime SHA mismatch: expected {OPS_RUNTIME_SHA}, got {head}')
if dirty:
    raise RuntimeError(f'operator runtime checkout is dirty: {dirty}')

print('Pinned Track-A master runtime:', head)


In [ ]:
driver = OPS_ROOT / 'journal_extension/kaggle/tracka_v12_ops/master_account_driver_v5.py'
cp = subprocess.run([sys.executable, '-u', str(driver), 'K3'], cwd=driver.parent)

if cp.returncode == 0:
    print('K3 master: TERMINAL PASS for this account queue.')
elif cp.returncode == 2:
    print('K3 master: controlled dependency/session/publication continuation. Rerun THIS SAME notebook as a fresh Batch version.')
else:
    raise RuntimeError(f'K3 master requires investigation; rc={cp.returncode}')


Recovery rule: do not create a separate recovery notebook. A controlled rollover is resumed by **Save Version -> Save & Run All** on this same account notebook. Actual validation/scientific failures remain fail-closed.
